In [38]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, set_default_openai_client,set_tracing_disabled,set_tracing_export_api_key
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from openai import AsyncOpenAI

load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
set_default_openai_client(client)
model = "llama-3.3-70b-versatile"

In [45]:
set_tracing_disabled(False)

In [47]:
set_tracing_export_api_key(os.environ.get("OPENAI_API_KEY"))

In [ ]:
# Regular function for testing (no decorator)
def send_email_impl(subject: str, html_body: str) -> Dict[str, str]:
    """
    Send an email with the given subject and HTML body.
    
    Args:
        subject: The subject line of the email
        html_body: The HTML content of the email body
    """
    try:
        sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
        mail = Mail(
            from_email=Email("jay.k@crestskillserve.com"),
            to_emails=To("jaygdsc0@gmail.com"),
            subject=subject,
            html_content=Content("text/html", html_body)
        )
        response = sg.client.mail.send.post(request_body=mail.get())
        print(f"SendGrid response: {response.status_code}")
        return {"status": "sent", "code": response.status_code}
    except Exception as e:
        print(f"SendGrid error: {e}")
        return {"status": "error", "message": str(e)}

# Create the tool version for the agent
send_email_tool = function_tool(send_email_impl)    

In [40]:
agent_pro = Agent(name="Pro", instructions="Write professional cold emails for ComplAI (SOC2 compliance SaaS). Keep it under 200 words.", model=model)
agent_fun = Agent(name="Fun", instructions="Write witty cold emails for ComplAI. Use humor but stay professional. Under 200 words.", model=model)
agent_fast = Agent(name="Fast", instructions="Write concise cold emails for ComplAI. Get to the point fast. Under 150 words.", model=model)

pro_tool = agent_pro.as_tool(tool_name="pro_agent", tool_description="Professional email")
fun_tool = agent_fun.as_tool(tool_name="fun_agent", tool_description="Witty email")
fast_tool = agent_fast.as_tool(tool_name="fast_agent", tool_description="Concise email")

In [41]:
subject_agent = Agent(name="Subject", instructions="Write a catchy subject line for a cold email. Max 8 words.", model=model)
html_agent = Agent(name="HTML", instructions="Convert plain text to simple HTML email. Add <p>, <br>, <b> tags. Keep it clean.", model=model)

subject_tool = subject_agent.as_tool(tool_name="subject_writer", tool_description="Write subject")
html_tool = html_agent.as_tool(tool_name="html_converter", tool_description="Convert to HTML")

In [42]:
email_agent = Agent(
    name="EmailManager",
    instructions="""You are the Email Manager. You MUST process every email you receive.

The email text to process is in the last message from the SalesManager. Use that email text.

STEPS YOU MUST FOLLOW:
1. Take that email text
2. Call subject_writer with that email text
3. Wait for the subject
4. Call html_converter with that email text
5. Wait for the HTML
6. Call send_email with the subject and HTML

FAILURE IS NOT AN OPTION. You MUST call all three tools in order.""",
    tools=[subject_tool, html_tool, send_email_tool],  # Changed to send_email_tool
    model=model
)

In [43]:
# Cell 21 (SalesManager – updated instructions)
sales_agent = Agent(
    name="SalesManager",
    instructions="""Call all three agents:
1. pro_agent: "Write email to Dear CEO from Alice"
2. fun_agent: "Write email to Dear CEO from Alice"
3. fast_agent: "Write email to Dear CEO from Alice"

Pick the best one. Output the chosen email text, then immediately transfer to email_agent to process and send it.""",
    tools=[pro_tool, fun_tool, fast_tool],
    handoffs=[email_agent],
    model=model
)

In [50]:
with trace("Solution"):
    result = await Runner.run(sales_agent, "Send email")
    print(" Email sent! Check inbox.")

SendGrid response: 202
 Email sent! Check inbox.


In [48]:
print("RESULT:", result.final_output)
print("TOOLS IN EMAIL AGENT:", [t.name for t in email_agent.tools])

RESULT: The email with the subject "ComplAI: Unlock Business Potential" has been sent successfully.
TOOLS IN EMAIL AGENT: ['subject_writer', 'html_converter', 'send_email_impl']
